In [1]:
import numpy as np
import pandas as pd

In [2]:
%run ANN.ipynb
%run LinearRegression.ipynb

New X_train_prepared shape: (8392, 12)
New y_train shape: (8392,)
Epoch 0, Loss: 32.16567356870645
Epoch 100, Loss: 27.976318209159615
Epoch 200, Loss: 24.543539867245293
Epoch 300, Loss: 21.7214504052137
Epoch 400, Loss: 19.380990098462856
Epoch 500, Loss: 17.394196161681577
Epoch 600, Loss: 15.617465624246279
Epoch 700, Loss: 13.90661699438549
Epoch 800, Loss: 12.243673202207834
Epoch 900, Loss: 10.85989369975853
Epoch 1000, Loss: 9.931020290799475
Epoch 1100, Loss: 9.329303372946757
Epoch 1200, Loss: 8.884904879404917
Epoch 1300, Loss: 8.522793705719923
Epoch 1400, Loss: 8.215840370374009
Epoch 1500, Loss: 7.951625921948878
Epoch 1600, Loss: 7.751272582305104
Epoch 1700, Loss: 7.608103369398435
Epoch 1800, Loss: 7.502612767518588
Epoch 1900, Loss: 7.425255186990435
Epoch 2000, Loss: 7.368183995912431
Epoch 2100, Loss: 7.325058925382066
Epoch 2200, Loss: 7.291175743445583
Epoch 2300, Loss: 7.265214194306581
Epoch 2400, Loss: 7.24511533300361
Epoch 2500, Loss: 7.231340916949415
Epoch 

In [3]:
# X_train_prepared 
# X_test_prepared

#ZSCORE
X_train_data_Z, X_validate_data_Z = z_score(X_train_prepared, X_test_prepared)

#Bias
X_bias_train_data = np.hstack([np.ones((X_train_data_Z.shape[0], 1)), X_train_data_Z])
X_bias_validate_data = np.hstack([np.ones((X_validate_data_Z.shape[0], 1)), X_validate_data_Z])

#Weights
XtX = X_bias_train_data.T @ X_bias_train_data
XtY = X_bias_train_data.T @ y_train

weight = np.linalg.pinv(XtX) @ XtY

In [4]:
def weighted_mean_voting(lr_pred, ann_pred, lr_weight, ann_weight):
    
    lr_pred = lr_pred.reshape(-1, 1)
    ann_pred = ann_pred.reshape(-1, 1)

    ensemble_pred = (lr_weight * lr_pred) + (ann_weight * ann_pred)

    return ensemble_pred

In [5]:
# Predict
lr_train_pred = X_bias_train_data @ weight
lr_test_pred = X_bias_validate_data @ weight

ann_train_pred = np.expm1(model.forward_prop(X_train_prepared))
ann_test_pred = np.expm1(model.forward_prop(X_test_prepared))

# Train
ensemble_train_pred = weighted_mean_voting(
    lr_train_pred,
    ann_train_pred,
    lr_weight=0.7,
    ann_weight=0.3
)

ensemble_test_pred = weighted_mean_voting(
    lr_test_pred,
    ann_test_pred,
    lr_weight=0.7,
    ann_weight=0.3
)

# Calculate RMSE (Root Mean Squared Error)
ensemble_train_rmse = np.sqrt(np.mean((y_train - ensemble_train_pred) ** 2))
ensemble_test_rmse = np.sqrt(np.mean((y_test - ensemble_test_pred) ** 2))

# Calculate sMAPE
ensemble_train_smape = smape(y_train, ensemble_train_pred)
ensemble_test_smape = smape(y_test, ensemble_test_pred)

print(f"Train RMSE: {train_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")
print(f"Train sMAPE: {train_smape_val:.2f}%")
print(f"Test sMAPE: {test_smape_val:.2f}%")

Train RMSE: 1565160021.03
Test RMSE: 1682495424.48
Train sMAPE: 135.98%
Test sMAPE: 136.59%
